In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
import os, time

def scrape(path, headless=False):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--hide-scrollbars")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.get(path)
    driver.maximize_window()

    # Login Steps
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'username'))).send_keys("fourbrotherstrading@icloud.com")
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'password'))).send_keys("Sultanmirza1501#")
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, './/button[text()="Sign in"]'))).click()
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, 'start-btn'))).click()
    except:
        print("Already logged in / No popup")

    return driver


url = "https://www.centralcarauctions.com/portal/auction/buyer/webauction/auction/864"
driver = scrape(url, headless=False)

os.makedirs("live_html", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

previous_title = ""
file_counter = 1

try:
    while True:
        if not driver.window_handles:
            print("Browser closed. Exiting.")
            break

        try:
            # 👉 Corrected XPATH (col-lg-9)
            try:
                car_title_elem = WebDriverWait(driver, 4).until(
                    EC.presence_of_element_located((By.XPATH, './/div[contains(@class,"col-lg-9")]/h1'))
                )
                current_title = car_title_elem.text.strip()
            except TimeoutException:
                print("🔄 Waiting for car data...")
                time.sleep(1)
                continue

            if current_title != previous_title:
                previous_title = current_title
                print(f"\n📌 NEW VEHICLE: {current_title}")

                # Extract Registration
                try:
                    reg_element = WebDriverWait(driver, 3).until(
                        EC.presence_of_element_located((
                            By.XPATH,
                            '//table[contains(@class,"vehicle-table")]//th[text()="Registration"]/following-sibling::td'
                        ))
                    )
                    reg_number = reg_element.text.strip().replace(" ", "_")
                except:
                    reg_number = current_title[:10].replace(" ", "_")

                # FILE NAMES
                html_file = os.path.join("live_html", f"{file_counter}_{reg_number}.html")
                screenshot_file = os.path.join("screenshots", f"{file_counter}_{reg_number}.png")

                # SAVE HTML
                with open(html_file, "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"💾 HTML SAVED: {html_file}")

                # SAVE SCREENSHOT
                driver.save_screenshot(screenshot_file)
                print(f"📸 SCREENSHOT SAVED: {screenshot_file}")

                file_counter += 1
            
            time.sleep(2)

        except StaleElementReferenceException:
            print("♻️ Stale element — retrying...")
            continue

        except Exception as e:
            print("⚠️ Waiting (temporary issue)...")
            time.sleep(2)
            continue

except KeyboardInterrupt:
    print("⛔ Stopped by user.")
finally:
    driver.quit()
    print("🚪 Browser closed.")


Already logged in / No popup
Browser closed. Exiting.
🚪 Browser closed.


In [2]:
import os
import csv
from bs4 import BeautifulSoup

def parse_last_html():
    folder = "live_html"
    files = sorted(os.listdir(folder), key=lambda x: os.path.getmtime(os.path.join(folder, x)))
    last_file = os.path.join(folder, files[-1])
    
    print(f"📌 Reading File: {last_file}")
    
    with open(last_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    bid_list = soup.find("ul", id="biddinghistory")
    if not bid_list:
        print("⚠️ No Bidding History Found!")
        return

    items = bid_list.find_all("li")

    results = []
    current_lot = None
    bids = []
    status = ""

    for li in items:
        text = li.get_text(strip=True).lower()
        raw = li.get_text(strip=True)

        if "lot changed:" in text:
            if current_lot:
                last_bid = bids[0] if bids else ""
                results.append([current_lot, bids, status, last_bid])
            current_lot = raw.replace("Lot changed:", "").strip()
            bids = []
            status = ""

        elif "not sold" in text:
            status = "not_sold"

        elif "provisionally" in text and "sold" in text:
            status = "provisionally"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "sold" in text and "not sold" not in text:
            status = "sold"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "progress" in text:
            status = "in_progress"

        elif "bid:" in text:
            if status in ["sold", "provisionally"]:
                price = raw.split("£")[-1].strip()
                bids.append("£" + price)

    # Last lot
    if current_lot:
        last_bid = bids[0] if bids else ""
        results.append([current_lot, bids, status, last_bid])

    # Merge empty lots into previous
    cleaned_results = []
    for lot, bids_list, st, last_bid in results:
        if bids_list or st in ["sold", "provisionally"]:  # keep if it has bids or sold/provisional
            cleaned_results.append([lot, bids_list, st, last_bid])
        else:
            # merge status if previous exists
            if cleaned_results:
                cleaned_results[-1][2] = st  # update status of previous lot

    # Flatten bids list to string
    final_results = []
    for lot, bids_list, st, last_bid in cleaned_results:
        bids_str = ", ".join(bids_list) if bids_list else ""
        final_results.append([lot, bids_str, st, last_bid])

    # Print output
    print("\n=== BIDDING RESULT ===")
    for row in final_results:
        print(row)

    # Save CSV
    csv_file = "CCA_LIVE_data.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot", "Bids", "Bidding Status", "Last Bid"])
        for row in final_results:
            writer.writerow(row)

    print(f"\n💾 CSV Saved Successfully: {csv_file}")


# RUN
parse_last_html()


📌 Reading File: live_html\110_BP16UOC.html

=== BIDDING RESULT ===
['105', '£850, £850, £800, £750, £700, £600, £500', 'sold', '£850']
['104', '£2,400, £2,400, £2,375, £2,350, £2,300, £2,200, £2,100, £2,000', 'sold', '£2,400']
['103', '£2,550, £2,550, £2,500, £2,400, £2,300, £2,200', 'sold', '£2,550']
['102', '£2,550, £2,550, £2,500, £2,450, £2,400, £2,300, £2,200, £2,100, £2,000', 'sold', '£2,550']
['101', '£4,400, £4,400, £4,300, £4,200, £4,100, £4,000, £3,900, £3,800', 'provisionally', '£4,400']
['100', '£4,500, £4,500, £4,400, £4,300, £4,200, £4,100, £4,000', 'sold', '£4,500']
['99', '£6,500, £6,500, £6,400, £6,300, £6,200, £6,100, £6,000, £5,900, £5,800, £5,700', 'sold', '£6,500']
['98', '£3,100, £3,100, £3,000, £2,900, £2,800, £2,700, £2,600, £2,500', 'sold', '£3,100']
['97', '£4,700, £4,700, £4,650, £4,600, £4,550, £4,500, £4,450, £4,400, £4,350, £4,300, £4,200, £4,100, £4,000, £3,900, £3,800, £3,700, £3,600', 'sold', '£4,700']
['96', '£9,500, £9,500, £9,400, £9,200, £9,000', 'p

In [3]:
import os
import csv

def save_lot_reg_csv():
    folder = "live_html"
    files = sorted(os.listdir(folder), key=lambda x: int(x.split("_")[0])) 

    csv_file = "lot_reg_mapping.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot Number", "Reg"])  # header

        for file in files:
            lot_number = file.split("_")[0]
            reg_name = file.split("_")[1].replace(".html", "")
            writer.writerow([lot_number, reg_name])

    print(f"\n💾 CSV Saved Successfully: {csv_file}")

# RUN
save_lot_reg_csv()



💾 CSV Saved Successfully: lot_reg_mapping.csv


In [4]:
import pandas as pd
import os


live_df = pd.read_csv("CCA_LIVE_data.csv")
reg_df = pd.read_csv("lot_reg_mapping.csv")


reg_df["Lot Number"] = reg_df["Lot Number"].astype(int)
live_df = live_df.reset_index(drop=True)


live_df["Reg"] = reg_df["Reg"].tolist()[:len(live_df)]  


final_csv = "CCA_LIVE_data_final.csv"
live_df.to_csv(final_csv, index=False)

os.remove("CCA_LIVE_data.csv")
os.remove("lot_reg_mapping.csv")

print(f"💾 Merged CSV saved as {final_csv} and original files deleted.")


💾 Merged CSV saved as CCA_LIVE_data_final.csv and original files deleted.


In [ ]:
import pandas as pd

# Load CSVs
live_df = pd.read_csv("CCA_LIVE_data_final.csv")
reg_df = pd.read_csv("cca_data.csv")

# Ensure Reg columns are string type
live_df['Reg'] = live_df['Reg'].astype(str)
reg_df['Reg'] = reg_df['Reg'].astype(str)

# Keep only relevant columns from live_df and rename Bids -> Bidding History
live_df = live_df[["Bids", "Bidding Status", "Last Bid", "Reg"]].rename(columns={"Bids": "Bidding History"})

# Merge on Reg, keep all rows from reg_df
merged_df = pd.merge(reg_df, live_df, on="Reg", how="left")  # merge live data at the end

# Remove duplicates if any
merged_df = merged_df.drop_duplicates(subset="Reg")

# Save final CSV
merged_df.to_csv("final_cca.csv", index=False)

print(f"💾 final_caa.csv created successfully! Total records: {len(merged_df)}")


💾 final_caa.csv created successfully! Total records: 60
